# Solution 01: label with SAM, train YOLO, check it for the robot
One way to do tasks 1-4. Try them yourself first; compare afterwards.
Needs a GPU for task 3 (two YOLO trainings, ~3 min each on a T4). Task 4 advanced needs
`pip install onnx onnxslim onnxruntime`.

In [ ]:
# Setup: installs what is missing (Colab / RunPod) and finds vision.py
import importlib.util, pathlib, subprocess, sys, time, urllib.request
for pkg, mod in [("transformers", "transformers"), ("ultralytics", "ultralytics"),
                 ("segmentation-models-pytorch", "segmentation_models_pytorch")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
if (pathlib.Path.cwd().parent / "vision.py").exists():   # opened from vision/notebooks or vision/sol
    sys.path.insert(0, str(pathlib.Path.cwd().parent))
elif not pathlib.Path("vision.py").exists():            # Colab: fetch it from the course repo
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/zhaw-physical-ai/cas-pair/main/vision/vision.py", "vision.py")
from vision import *
device = get_device()
print("device:", device)

In [ ]:
images, instances, train, test = load_pennfudan()
test_images = [images[k] for k in test]
test_instances = [instances[k] for k in test]
sam = SamLabeler()      # SAM 2.1 base-plus, ~300 MB on first use

## Given helpers
From `vision.py`; read the docstrings with `help(...)`:
- `SamLabeler`: `sam(image, boxes=...)` or `sam(image, points=..., point_labels=...)` returns masks and SAM's own score
- `boxes_from_instances`, `instances_to_masks`: boxes and per-person masks from the human labels
- `iou`, `mean_iou`, `person_recall`, `seconds_per_image`: metrics
- `write_yolo_dataset`, `train_yolo`, `yolo_predict`: YOLO plumbing
- `UNetSegmenter`: a semantic U-Net (task 3, advanced)
- `show(ax, image, mask, boxes=..., points=..., point_labels=...)`: plotting

One SAM call, as an example:

In [ ]:
i = train[0]
boxes = boxes_from_instances(instances[i])
masks, scores = sam(images[i], boxes=boxes)
print(masks.shape, "| SAM's own guess:", scores.round(2), "| IoU vs human:", round(iou(masks.any(0), instances[i] > 0), 3))
fig, ax = plt.subplots(figsize=(6, 6))
show(ax, images[i], masks, boxes=boxes, title="SAM 2.1 from boxes")

## Task 1: SAM as a labeling tool
Pick 3-5 people from different images (`instances[i] == pid` is one person's human mask). Prompt
SAM with one click, then with a box, and compare each mask with the human mask.

In [ ]:
picks = [(train[k], 1) for k in (0, 5, 10, 20, 30)]      # (image, person id)
fig, axs = plt.subplots(len(picks), 3, figsize=(12, 4 * len(picks)))
scores = []
for row, (i, pid) in zip(axs, picks):
    person = instances[i] == pid
    ys, xs = np.nonzero(person)
    click = [np.median(xs), np.median(ys)]
    box = boxes_from_instances(instances[i])[pid - 1]
    m_click = sam(images[i], points=[click])[0][0]
    m_box = sam(images[i], boxes=[box])[0][0]
    scores.append((iou(m_click, person), iou(m_box, person)))
    show(row[0], images[i], person, title=f"human, image {i}")
    show(row[1], images[i], m_click, points=[click], title=f"1 click: IoU {scores[-1][0]:.2f}")
    show(row[2], images[i], m_box, boxes=[box], title=f"box: IoU {scores[-1][1]:.2f}")
plt.tight_layout()
print("mean IoU  click %.3f  box %.3f" % tuple(np.mean(scores, 0)))

Find a click that makes SAM segment the *wrong* thing (a bag, a shirt, two people), then fix
it with more clicks: positive (label 1) on what is missing, negative (label 0) on what is too much.

In [ ]:
# search: 5 clicks down the centre line of each person, keep the ones that go wrong
bad = []
for i in train[:40]:
    for pid in np.unique(instances[i])[1:]:
        person = instances[i] == pid
        ys, xs = np.nonzero(person)
        for f in (0.1, 0.3, 0.5, 0.7, 0.9):
            y = ys.min() + f * np.ptp(ys)
            row = xs[np.abs(ys - y) < 2]                      # person pixels on that line
            if len(row):
                x = row[np.argmin(np.abs(row - np.median(row)))]  # a real person pixel, not the gap between the legs
                click = [float(x), float(y)]
                score = iou(sam(images[i], points=[click])[0][0], person)
                if score < 0.5:
                    bad.append((score, i, pid, click))
bad.sort()
print(f"{len(bad)} bad clicks out of ~{5 * sum(len(np.unique(instances[i])) - 1 for i in train[:40])}")
fig, axs = plt.subplots(1, 4, figsize=(20, 5))
for ax, (score, i, pid, click) in zip(axs, bad[::max(1, len(bad) // 4)]):
    show(ax, images[i], sam(images[i], points=[click])[0][0], points=[click], title=f"IoU {score:.2f}")
plt.tight_layout()

In [ ]:
# fix the worst one the way an annotator would: click on the biggest error, 3 times
score, i, pid, click = bad[0]
person = instances[i] == pid
points, labels = [click], [1]
m = sam(images[i], points=points)[0][0]
fig, axs = plt.subplots(1, 4, figsize=(20, 5))
show(axs[0], images[i], m, points=points, title=f"1 click: IoU {iou(m, person):.2f}")
for ax in axs[1:]:
    missing, extra = person & ~m, m & ~person
    err, lab = (missing, 1) if missing.sum() > extra.sum() else (extra, 0)
    ys, xs = np.nonzero(err)
    k = np.argmin((ys - np.median(ys)) ** 2 + (xs - np.median(xs)) ** 2)    # an error pixel near its centre
    points.append([float(xs[k]), float(ys[k])]); labels.append(lab)
    m = sam(images[i], points=points, point_labels=labels)[0][0]
    show(ax, images[i], m, points=points, point_labels=labels, title=f"{len(points)} clicks: IoU {iou(m, person):.2f}")
plt.tight_layout()

**Result:** A box is the most reliable prompt (IoU ~0.9). One click is often just as good, but on average clearly worse (~0.7 here), because some clicks go wrong: SAM returns only the bag, the jacket, the head, or two people standing together. A click does not say *how much* of the object you mean; a box does. A few extra positive or negative clicks usually fix it. For 10 000 images: boxes, because they are fast to draw and leave SAM little to guess.

### Advanced: sloppy boxes

In [ ]:
def sloppy(box, f, shape, rng):
    """Grow each side of the box by a random 0..f of its width / height (clipped to the image)."""
    x0, y0, x1, y1 = box
    d = rng.uniform(0, f, 4) * [x1 - x0, y1 - y0, x1 - x0, y1 - y0]
    return [max(0, x0 - d[0]), max(0, y0 - d[1]), min(shape[1] - 1, x1 + d[2]), min(shape[0] - 1, y1 + d[3])]

for f in (0, 0.1, 0.25, 0.5):
    rng = np.random.default_rng(0)
    s = [iou(sam(images[i], boxes=[sloppy(b, f, images[i].shape, rng) for b in boxes_from_instances(instances[i])])[0].any(0),
             instances[i] > 0) for i in train[:30]]
    print(f"boxes up to {f:.0%} too large: mean IoU {np.mean(s):.3f}")

## Task 2: Label the training set
Turn the human **boxes** of all 120 training images into SAM masks: these are your labels. Keep
one `(n, H, W)` array per image, because YOLO needs one mask per person.

In [ ]:
t0 = time.time()
sam_labels = [sam(images[i], boxes=boxes_from_instances(instances[i]))[0] for i in train]
sam_seconds = (time.time() - t0) / len(train)
label_iou = np.array([iou(m.any(0), instances[i] > 0) for m, i in zip(sam_labels, train)])
print(f"{sam_seconds:.2f} s per image | mean IoU {label_iou.mean():.3f} | below 0.8: {(label_iou < 0.8).sum()} images")

Look at the 5 worst images. Who is wrong, SAM or the human?

In [ ]:
worst = np.argsort(label_iou)[:5]
fig, axs = plt.subplots(2, 5, figsize=(25, 10))
for col, k in enumerate(worst):
    i = train[k]
    show(axs[0, col], images[i], instances[i], title=f"human, image {i}")
    show(axs[1, col], images[i], sam_labels[k], title=f"SAM, IoU {label_iou[k]:.2f}")
plt.tight_layout()

**Result:** SAM's labels agree with the human masks at IoU ~0.93 on average; only a handful of images are below 0.8. In the worst images the two disagree about what a person *is*: the human included bags and umbrellas, SAM left them out; SAM is often tighter around hair and between the legs, where the human polygon is coarse. Neither is simply wrong: a labeling guideline would have to decide. Good enough to train on, which task 3 checks.

### Advanced: a smaller SAM

In [ ]:
sam_tiny = SamLabeler("facebook/sam2.1-hiera-tiny")
for name, s in [("base-plus", sam), ("tiny", sam_tiny)]:
    t0 = time.time()
    q = [iou(s(images[i], boxes=boxes_from_instances(instances[i]))[0].any(0), instances[i] > 0) for i in train[:30]]
    print(f"{name:10s} mean IoU {np.mean(q):.3f}  {(time.time() - t0) / 30:.2f} s per image")
del sam_tiny

## Task 3: Train your own YOLO
Write your SAM labels as a YOLO dataset and fine-tune `yolo11n-seg`, starting from weights
pretrained on COCO (~2-3 min on a T4):
```python
yaml = write_yolo_dataset("yolo_sam", [images[i] for i in train], sam_labels)
yolo_sam = train_yolo(yaml, name="sam", epochs=30)     # Ultralytics underneath; see help(train_yolo)
```
Then measure it on the 50 test images against the human masks:
`mean_iou(lambda im: yolo_predict(yolo_sam, im)[0].any(0), test_images, test_instances)`.

In [ ]:
def fit_yolo(name, masks, idx=train, epochs=30):
    yaml = write_yolo_dataset(f"yolo_{name}", [images[i] for i in idx], masks)
    return train_yolo(yaml, name=name, epochs=epochs)

def yolo_mask(model, conf=0.25):
    return lambda im: yolo_predict(model, im, conf=conf)[0].any(0)

yolo_sam = fit_yolo("sam", sam_labels)
iou_sam = mean_iou(yolo_mask(yolo_sam), test_images, test_instances)
print(f"YOLO on SAM labels: test IoU {iou_sam:.3f}")

In [ ]:
yolo_human = fit_yolo("human", [instances_to_masks(instances[i]) for i in train])
iou_human = mean_iou(yolo_mask(yolo_human), test_images, test_instances)
print(f"YOLO on human labels: test IoU {iou_human:.3f}   (on SAM labels: {iou_sam:.3f})")

Look at test images with several people. What can a robot do with one mask per person that
"these pixels are person" cannot give it?

In [ ]:
crowded = np.argsort([-len(np.unique(m)) for m in test_instances])[:4]
fig, axs = plt.subplots(2, 4, figsize=(24, 10))
for col, k in enumerate(crowded):
    masks, boxes, conf = yolo_predict(yolo_sam, test_images[k])
    show(axs[0, col], test_images[k], test_instances[k], boxes=boxes_from_instances(test_instances[k]),
         title=f"human: {len(np.unique(test_instances[k])) - 1} people")
    show(axs[1, col], test_images[k], masks, boxes=boxes, title=f"YOLO: {len(boxes)} people")
plt.tight_layout()

**Result:** YOLO trained on SAM labels reaches a test IoU of ~0.86, the same as YOLO trained on the human masks (within a few hundredths; which one wins changes with the run). So the boxes-plus-SAM labels were as useful as the ten times more expensive masks. One reason: the test score is measured against the human masks, so part of the remaining error is the humans' own style (bags, coarse outlines). YOLO gives one mask and box per person, so a robot can count people, track each one over time, and estimate where each one stands. It misses small people in the background and sometimes merges two people who overlap.

### Advanced: a semantic U-Net on the same labels

In [ ]:
t0 = time.time()
unet = UNetSegmenter().fit([images[i] for i in train], [m.any(0) for m in sam_labels])
print(f"U-Net trained in {time.time() - t0:.0f} s: test IoU {mean_iou(unet, test_images, test_instances):.3f}"
      f"  (YOLO: {iou_sam:.3f})")

## Task 4: Is it ready for the robot?
A perception-planning-control loop runs at 10-30 Hz. And IoU hides what matters for safety: was
the person seen at all? `person_recall` counts the people of whom at least half the pixels were
found, and lists the ones it missed.

In [ ]:
t_yolo = seconds_per_image(yolo_mask(yolo_sam), test_images)
print(f"SAM 2.1 (needs a box per person!)  {1000 * sam_seconds:6.1f} ms  {1 / sam_seconds:5.1f} Hz")
print(f"YOLO                               {1000 * t_yolo:6.1f} ms  {1 / t_yolo:5.1f} Hz")

for conf in (0.1, 0.25, 0.5):
    rec, misses = person_recall(yolo_mask(yolo_sam, conf), test_images, test_instances)
    false_px = np.mean([(yolo_mask(yolo_sam, conf)(im) & (inst == 0)).mean() for im, inst in zip(test_images, test_instances)])
    print(f"conf {conf:4.2f}: recall {rec:.3f} ({len(misses)} people missed), {false_px:.2%} of the background called person")

In [ ]:
rec, misses = person_recall(yolo_mask(yolo_sam, 0.25), test_images, test_instances)
fig, axs = plt.subplots(1, min(6, len(misses)), figsize=(24, 5), squeeze=False)
for ax, (k, pid) in zip(axs[0], misses):
    show(ax, test_images[k], yolo_mask(yolo_sam)(test_images[k]), boxes=[boxes_from_instances(test_instances[k])[pid - 1]],
         title=f"missed (green box), test image {k}")
plt.tight_layout()

**Result:** SAM takes ~250 ms per image on a GPU *and* needs a human to draw the boxes: it cannot sit in the loop. YOLO takes ~10-20 ms, 50+ Hz on a GPU, well within 10-30 Hz. A lower confidence finds more people (higher recall) but also calls more of the background a person, so the robot would stop for nothing more often. For a safety stop you would take the low threshold and accept the false stops. The missed people are mostly small (far away), cut by the image edge, or hidden behind someone else: exactly the cases from 00_data. Whether that is acceptable depends on where they are: a far-away person matters less than one in front of the robot.

### Advanced: ONNX on the CPU, and a stop rule

In [ ]:
from ultralytics import YOLO
onnx_path = yolo_sam.export(format="onnx", imgsz=640)
yolo_onnx = YOLO(onnx_path, task="segment")
t_cpu = seconds_per_image(lambda im: yolo_predict(yolo_onnx, im, device="cpu")[0], test_images[:20])
print(f"YOLO ONNX on the CPU: {1000 * t_cpu:.0f} ms per image, {1 / t_cpu:.1f} Hz")

def stop(masks, frac=0.6):
    """Stop if any person is taller than frac of the image height: close to the camera."""
    return any(np.ptp(np.nonzero(m.any(1))[0]) > frac * m.shape[0] for m in masks if m.any())

pred = np.array([stop(yolo_predict(yolo_sam, im)[0]) for im in test_images])
true = np.array([stop(instances_to_masks(inst)) for inst in test_instances])
print(f"stops needed {true.sum()}/{len(true)}, made {(pred & true).sum()}, missed {(~pred & true).sum()}, "
      f"for nothing {(pred & ~true).sum()}")

## Report
Add one row per model to the class results table: labels (SAM / human), model, training images,
test IoU, `person_recall` at your chosen threshold, ms per image, GPU.